# ISA-PHM Protocol Analysis

This notebook extracts and displays measurement and processing protocols from ISA-JSON format, showing protocol parameters mapped to sensors in grid format.

In [1]:
import json
import pandas as pd

# Configure pandas display to prevent wrapping
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.expand_frame_repr', False)

## Load ISA-JSON Data

In [2]:
# Display processing protocols
df_proc = create_protocol_grid(0, 'processing')

NameError: name 'create_protocol_grid' is not defined

In [ ]:
# Example: Study 2
# df_meas_s2 = create_protocol_grid(1, 'measurement')
# df_proc_s2 = create_protocol_grid(1, 'processing')

## Analyze Other Studies

You can analyze any study by changing the study index (0-15 for the 16 studies in this dataset):

In [ ]:
# Display measurement protocols
df_meas = create_protocol_grid(0, 'measurement')


MEASUREMENT PROTOCOLS - Study 1: Case 1
                       Vibration 1 Vibration 2             Current 1             Current 2 Acoustic Emission 1 Acoustic Emission 2
Sensor Location              Table     Spindle                     -                     -               Table             Spindle
Sensor Frequency Range      13 kHz      13 kHz                     -                     -               2 MHz               2 MHz
Filter Type              High pass   High pass                     -                     -                   -                   -
HP cutoff frequency          8 kHz       8 kHz                     -                     -                   -                   -
Amplifier                  DE 302A     DE 302A                     -                     -              DE302A              DE302A
Amplifier max load             5 V         5 V                     -                     -                 5 V                 5 V
Amplifier min load            -5 V        

## Display Protocol Grids for Study 1

In [ ]:
def create_protocol_grid(study_idx=0, protocol_type='measurement'):
    """
    Extract protocol parameters and create a grid showing Parameter × Sensor mappings.
    
    Args:
        study_idx: Index of the study (0-based)
        protocol_type: 'measurement' or 'processing'
    
    Returns:
        pandas DataFrame with parameters as rows and sensors as columns
    """
    study = data['studies'][study_idx]
    
    # Build protocol lookup: protocol @id -> protocol info
    protocol_lookup = {}
    for protocol in study.get('protocols', []):
        protocol_id = protocol.get('@id', '')
        protocol_name = protocol.get('name', '')
        
        # Build parameter lookup for this protocol: parameter @id -> parameter name
        param_lookup = {}
        for param in protocol.get('parameters', []):
            param_id = param.get('@id', '')
            param_name = param.get('parameterName', {}).get('annotationValue', 'Unknown')
            param_lookup[param_id] = param_name
        
        protocol_lookup[protocol_id] = {
            'name': protocol_name,
            'params': param_lookup
        }
    
    # Determine which protocols to include based on type
    if protocol_type == 'measurement':
        # Measurement protocols contain "measurement" in their name
        target_protocols = {pid: info for pid, info in protocol_lookup.items() 
                          if 'measurement' in info['name'].lower()}
    else:  # processing
        # Processing protocols contain "processing" in their name
        target_protocols = {pid: info for pid, info in protocol_lookup.items() 
                          if 'processing' in info['name'].lower()}
    
    # Collect data from all assays in this study
    data_dict = {}  # {parameter_name: {sensor_name: value_with_unit}}
    sensors = []
    sensor_counter = 1
    
    for assay in study.get('assays', []):
        # Simple sensor naming: Sensor 1, Sensor 2, etc.
        sensor_name = f"Sensor {sensor_counter}"
        sensors.append(sensor_name)
        sensor_counter += 1
        
        # Process each process in the sequence
        for process in assay.get('processSequence', []):
            # Check if this process uses one of our target protocols
            protocol_ref = process.get('executesProtocol', {}).get('@id', '')
            
            if protocol_ref not in target_protocols:
                continue
            
            protocol_info = target_protocols[protocol_ref]
            
            # Extract parameter values
            for pv in process.get('parameterValues', []):
                # Get parameter name by resolving the category @id
                param_ref = pv.get('category', {}).get('@id', '')
                param_name = protocol_info['params'].get(param_ref, 'Unknown Parameter')
                
                # Get value
                value = pv.get('value')
                
                # Get unit by resolving unit @id from study.unitCategories
                unit_ref = pv.get('unit', {})
                unit_term = ''
                if unit_ref and isinstance(unit_ref, dict):
                    unit_id = unit_ref.get('@id', '')
                    if unit_id:
                        resolved_unit = next((u for u in study.get('unitCategories', []) 
                                            if u.get('@id') == unit_id), None)
                        if resolved_unit:
                            unit_term = resolved_unit.get('annotationValue', '')
                
                # Format display value
                if value is not None:
                    if unit_term:
                        value_display = f"{value} {unit_term}"
                    else:
                        value_display = str(value)
                else:
                    value_display = '-'
                
                # Store in data structure
                if param_name not in data_dict:
                    data_dict[param_name] = {}
                data_dict[param_name][sensor_name] = value_display
    
    # Convert to DataFrame
    if not data_dict:
        print(f"\nNo {protocol_type} protocol parameters found for Study {study_idx + 1}")
        return pd.DataFrame()
    
    df = pd.DataFrame(data_dict).T
    df = df.reindex(columns=sensors)
    df = df.fillna('-')
    
    # Display
    print(f"\n{'='*100}")
    print(f"{protocol_type.upper()} PROTOCOLS - Study {study_idx + 1}: {study.get('title', 'Unknown')}")
    print(f"{'='*100}")
    print(df)
    
    return df

## Protocol Grid Function

This function extracts protocol parameters from the ISA-JSON structure and displays them in a grid format with sensors as columns.

In [ ]:
with open("isa-phm-out (35).json", "r") as f:
    data = json.load(f)

print(f"Loaded ISA-JSON with {len(data['studies'])} studies")
print(f"Investigation: {data['title']}")

Loaded ISA-JSON with 16 studies
Investigation: Tool Wear Dataset
